# 02 · 핵심 분석: 교차표·집단비교·회귀  (모듈 3)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/02_core_analysis.ipynb)

> 대표 분석 3종을 Python으로 수행한다. **같은 분석을 STATA로도** 하고(각 절의 🔁),
> 두 결과의 숫자가 같은지 **교차검증**한다. — "숫자는 같다. 출력 모양만 다르다."

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.formula.api as smf
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/sample_crs.csv")
df["log_disb"]  = np.log(df["USD_Disbursement"])
df["log_gdppc"] = np.log(df["RecipientGDPpc"])
df["log_pop"]   = np.log(df["RecipientPop"])
print("준비 완료:", df.shape)

## 1) 교차표(교차분석) — 공여국 × 분야별 평균 집행액

In [ ]:
pd.pivot_table(df, values="USD_Disbursement",
               index="DonorName", columns="SectorName",
               aggfunc="mean").round(1)

🔁 **STATA**: `table donorname sectorname, contents(mean usd_disbursement)`  ·  코드 → `stata/02_crosstab.do`

## 2) 집단 비교 — 무상 vs 유상(t검정), 분야 간(ANOVA)

In [ ]:
grant = df.loc[df.FinanceType == "Grant", "USD_Disbursement"]
loan  = df.loc[df.FinanceType == "Loan",  "USD_Disbursement"]
t, p = stats.ttest_ind(grant, loan, equal_var=False)  # Welch(분산 다를 때) → STATA는 ttest ..., unequal
print(f"[t검정] 무상 평균={grant.mean():.1f}  유상 평균={loan.mean():.1f}  t={t:.2f}  p={p:.2e}")

groups = [g["USD_Disbursement"].values for _, g in df.groupby("SectorName")]
F, p2 = stats.f_oneway(*groups)
print(f"[ANOVA] 분야 간 평균 차이  F={F:.2f}  p={p2:.2e}")

🔁 **STATA**: `ttest usd_disbursement, by(financetype) unequal`  ·  `oneway usd_disbursement sector_n`  ·  코드 → `stata/03_group_compare.do`

## 3) 회귀분석 — 배분 결정요인 (로그변환의 이유)
집행액은 **왜도가 크다**(소수 대형사업). 그대로 쓰면 분석이 왜곡되므로 **로그**를 취한다.
→ AI가 자동으로 안 해줄 수 있는 **설계 판단**이고, 이게 사람의 역할이다.

In [ ]:
print("집행액 왜도:", round(df["USD_Disbursement"].skew(), 2), "→ 큼 → 로그변환 필요")
m = smf.ols("log_disb ~ log_gdppc + log_pop", data=df).fit()
print(m.summary().tables[1])
print(f"\nR² = {m.rsquared:.3f}")

**해석**: `log_gdppc` 계수가 **음수** → 1인당 GDP가 낮은(가난한) 수원국일수록 배분액↑.
`log_pop` 계수가 **양수** → 인구가 많을수록 배분액↑.

🔁 **STATA**: `regress log_disb log_gdppc log_pop` — **한 줄**.  코드 → `stata/04_regression.do`
> 두 도구의 계수·p값이 같은지 **교차검증**: 같으면 신뢰, 다르면 하나가 틀린 것.

---
✅ **핵심**: 교차표·검정·회귀 — 입문자가 첫날 직접 돌렸다. 비결은 *문법 암기*가 아니라
*무엇을·왜 분석할지 설계*하고 *결과를 검증·해석*하는 것.